<a href="https://colab.research.google.com/github/quinzy7/swirl_courses/blob/master/sample_p2p_process_mining_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
!pip install pm4py plotly kaleido --quiet

import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

N_CASES          = 2_000          # realistic mid-market volume
START_DATE       = datetime(2023, 1, 1)
END_DATE         = datetime(2023, 12, 31)

COST_CENTRES     = ["CC-1010", "CC-2020", "CC-3030", "CC-4040", "CC-5050"]
VENDORS          = [f"VENDOR-{str(i).zfill(3)}" for i in range(1, 41)]
REQUESTERS       = [f"REQ-{str(i).zfill(3)}" for i in range(1, 21)]
AP_CLERKS        = [f"AP-{str(i).zfill(3)}"  for i in range(1, 11)]
APPROVERS        = [f"MGR-{str(i).zfill(3)}" for i in range(1, 6)]
SYSTEMS          = ["SAP-MM", "SAP-FI", "ARIBA", "MANUAL"]

#P2P activity sequence (correct path) ──
HAPPY_PATH = [
    "Create Purchase Requisition",
    "Approve Purchase Requisition",
    "Create Purchase Order",
    "Send Purchase Order to Vendor",
    "Receive Goods",
    "Receive Invoice",
    "Match Invoice to PO",
    "Approve Invoice",
    "Schedule Payment",
    "Execute Payment",
]

VARIANTS = [
    (
        "Compliant – Standard",
        HAPPY_PATH,
        0.45,
    ),
    (
        "Compliant – Fast Track (<€5k)",
        [
            "Create Purchase Requisition",
            "Create Purchase Order",          # auto-approved
            "Send Purchase Order to Vendor",
            "Receive Goods",
            "Receive Invoice",
            "Match Invoice to PO",
            "Approve Invoice",
            "Schedule Payment",
            "Execute Payment",
        ],
        0.20,
    ),
    (
        "Non-Compliant – Invoice Before GR",
        [
            "Create Purchase Requisition",
            "Approve Purchase Requisition",
            "Create Purchase Order",
            "Send Purchase Order to Vendor",
            "Receive Invoice",               # invoice arrives before goods
            "Receive Goods",
            "Match Invoice to PO",
            "Approve Invoice",
            "Schedule Payment",
            "Execute Payment",
        ],
        0.10,
    ),
    (
        "Non-Compliant – Maverick Buy",
        [
            "Receive Invoice",               # no PR or PO raised first
            "Create Purchase Order",         # retrospective PO
            "Match Invoice to PO",
            "Approve Invoice",
            "Schedule Payment",
            "Execute Payment",
        ],
        0.08,
    ),
    (
        "Bottleneck – Approval Delay",
        HAPPY_PATH,                          # same activities, longer gaps
        0.10,
    ),
    (
        "Rework – Invoice Mismatch",
        [
            "Create Purchase Requisition",
            "Approve Purchase Requisition",
            "Create Purchase Order",
            "Send Purchase Order to Vendor",
            "Receive Goods",
            "Receive Invoice",
            "Match Invoice to PO",
            "Reject Invoice",                # mismatch detected
            "Vendor Correction Requested",
            "Receive Invoice",               # second invoice received
            "Match Invoice to PO",
            "Approve Invoice",
            "Schedule Payment",
            "Execute Payment",
        ],
        0.07,
    ),
]

def pick_resource(activity: str) -> str:
    """
    Assign a realistic resource to each activity based on role.
    In a real SAP log, this maps to the 'org.resource' field.
    """
    if "Requisition" in activity:
        return random.choice(REQUESTERS)
    if "Approve" in activity or "Reject" in activity:
        return random.choice(APPROVERS)
    if "Purchase Order" in activity or "Send" in activity:
        return random.choice(AP_CLERKS)
    if "Invoice" in activity or "Match" in activity or "Payment" in activity or "Schedule" in activity:
        return random.choice(AP_CLERKS)
    if "Goods" in activity or "Vendor" in activity:
        return "SYSTEM"
    return "SYSTEM"

def pick_system(activity: str) -> str:
    """
    Map each activity to the system where it typically occurs.
    """
    mapping = {
        "Create Purchase Requisition":  "SAP-MM",
        "Approve Purchase Requisition": "SAP-MM",
        "Create Purchase Order":        "SAP-MM",
        "Send Purchase Order to Vendor":"ARIBA",
        "Receive Goods":                "SAP-MM",
        "Receive Invoice":              "SAP-FI",
        "Match Invoice to PO":          "SAP-FI",
        "Approve Invoice":              "SAP-FI",
        "Reject Invoice":               "SAP-FI",
        "Vendor Correction Requested":  "MANUAL",
        "Schedule Payment":             "SAP-FI",
        "Execute Payment":              "SAP-FI",
    }
    return mapping.get(activity, "SAP-FI")

def generate_activity_gaps(variant_label: str, n_activities: int) -> list:
    """
    Return a list of time deltas (in hours) between consecutive activities.
    Bottleneck variant deliberately inflates approval wait times.
    """
    if "Bottleneck" in variant_label:
        # Approval steps delayed by 3–10 business days
        gaps = [random.uniform(2, 8) for _ in range(n_activities)]
        # Spike gaps at approval positions (indices 1 and 7)
        for spike_idx in [1, 7]:
            if spike_idx < n_activities:
                gaps[spike_idx] = random.uniform(72, 240)
    elif "Maverick" in variant_label:
        gaps = [random.uniform(1, 24) for _ in range(n_activities)]
    elif "Rework" in variant_label:
        gaps = [random.uniform(4, 48) for _ in range(n_activities)]
        gaps[6] = random.uniform(48, 96)   # delay at mismatch
    else:
        gaps = [random.uniform(1, 36) for _ in range(n_activities)]
    return gaps

def simulate_p2p_log(n_cases: int) -> pd.DataFrame:
    """
    Simulate a realistic Purchase-to-Pay event log.

    Returns a DataFrame in XES-compatible format with columns:
        case_id, activity, timestamp, resource, system,
        cost_centre, vendor, invoice_amount, variant
    """
    records = []

    variant_labels  = [v[0] for v in VARIANTS]
    variant_acts    = [v[1] for v in VARIANTS]
    variant_weights = [v[2] for v in VARIANTS]

    for case_num in range(1, n_cases + 1):
        case_id = f"PO-2023-{str(case_num).zfill(5)}"

        # Pick a variant for this case
        variant_idx = random.choices(range(len(VARIANTS)), weights=variant_weights, k=1)[0]
        variant_label   = variant_labels[variant_idx]
        activities      = variant_acts[variant_idx]

        # Random case start scattered across the year
        total_seconds = int((END_DATE - START_DATE).total_seconds())
        case_start    = START_DATE + timedelta(seconds=random.randint(0, total_seconds))

        # Case-level attributes (constant across all events in a case)
        cost_centre    = random.choice(COST_CENTRES)
        vendor         = random.choice(VENDORS)
        invoice_amount = round(np.random.lognormal(mean=8.5, sigma=1.2), 2)  # realistic skewed distribution

        gaps = generate_activity_gaps(variant_label, len(activities))
        current_time = case_start

        for i, activity in enumerate(activities):
            if i > 0:
                current_time += timedelta(hours=gaps[i])

            records.append({
                "case_id":        case_id,
                "activity":       activity,
                "timestamp":      current_time,
                "resource":       pick_resource(activity),
                "system":         pick_system(activity),
                "cost_centre":    cost_centre,
                "vendor":         vendor,
                "invoice_amount": invoice_amount,
                "variant":        variant_label,
            })

    df = pd.DataFrame(records)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df.sort_values(["case_id", "timestamp"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df


print("Simulating P2P event log...")
event_log = simulate_p2p_log(N_CASES)

# ── Sanity checks ─────────────────────────────────────────────
print(f"\n✅ Event log generated")
print(f"   Total events  : {len(event_log):,}")
print(f"   Unique cases  : {event_log['case_id'].nunique():,}")
print(f"   Date range    : {event_log['timestamp'].min().date()} → {event_log['timestamp'].max().date()}")
print(f"   Activities    : {sorted(event_log['activity'].unique())}")
print(f"\nVariant distribution:")
print(event_log.groupby('variant')['case_id'].nunique().sort_values(ascending=False).to_string())
print(f"\nSample rows:")
event_log.head(15)

Simulating P2P event log...

✅ Event log generated
   Total events  : 19,416
   Unique cases  : 2,000
   Date range    : 2023-01-01 → 2024-01-16
   Activities    : ['Approve Invoice', 'Approve Purchase Requisition', 'Create Purchase Order', 'Create Purchase Requisition', 'Execute Payment', 'Match Invoice to PO', 'Receive Goods', 'Receive Invoice', 'Reject Invoice', 'Schedule Payment', 'Send Purchase Order to Vendor', 'Vendor Correction Requested']

Variant distribution:
variant
Compliant – Standard                 915
Compliant – Fast Track (<€5k)        396
Non-Compliant – Invoice Before GR    192
Non-Compliant – Maverick Buy         185
Bottleneck – Approval Delay          174
Rework – Invoice Mismatch            138

Sample rows:


,case_id,activity,timestamp,resource,system,cost_centre,vendor,invoice_amount,variant
0,PO-2023-00001,Create Purchase Requisition,2023-01-10 17:07:01.000000,REQ-001,SAP-MM,CC-3030,VENDOR-016,8920.05,Compliant – Fast Track (<€5k)
1,PO-2023-00001,Create Purchase Order,2023-01-11 19:53:36.372985,AP-009,SAP-MM,CC-3030,VENDOR-016,8920.05,Compliant – Fast Track (<€5k)
2,PO-2023-00001,Send Purchase Order to Vendor,2023-01-12 20:34:40.508400,AP-004,ARIBA,CC-3030,VENDOR-016,8920.05,Compliant – Fast Track (<€5k)
3,PO-2023-00001,Receive Goods,2023-01-14 04:48:15.133931,SYSTEM,SAP-MM,CC-3030,VENDOR-016,8920.05,Compliant – Fast Track (<€5k)
4,PO-2023-00001,Receive Invoice,2023-01-14 08:50:49.426842,AP-009,SAP-FI,CC-3030,VENDOR-016,8920.05,Compliant – Fast Track (<€5k)
5,PO-2023-00001,Match Invoice to PO,2023-01-15 00:36:51.576122,AP-007,SAP-FI,CC-3030,VENDOR-016,8920.05,Compliant – Fast Track (<€5k)
6,PO-2023-00001,Approve Invoice,2023-01-15 02:39:26.025771,MGR-002,SAP-FI,CC-3030,VENDOR-016,8920.05,Compliant – Fast Track (<€5k)
7,PO-2023-00001,Schedule Payment,2023-01-15 11:18:34.410596,AP-008,SAP-FI,CC-3030,VENDOR-016,8920.05,Compliant – Fast Track (<€5k)
8,PO-2023-00001,Execute Payment,2023-01-16 05:59:49.176897,AP-010,SAP-FI,CC-3030,VENDOR-016,8920.05,Compliant – Fast Track (<€5k)
9,PO-2023-00002,Create Purchase Requisition,2023-12-04 14:27:58.000000,REQ-015,SAP-MM,CC-1010,VENDOR-011,4163.38,Compliant – Standard


In [20]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful